# FoodmoleGPT Training Analysis

> Food Science Domain LLM — Ablation Study Training Report
>
> **Base Models**: Qwen3-8B-Base & Llama 3.1 8B | **Method**: LoRA (rank=64, alpha=128) | **Pipeline**: CPT + SFT

This notebook visualizes training loss curves and summarizes hyperparameters for all ablation experiments:
- **SFT-Only**: Direct supervised fine-tuning on food-science instruction data
- **CPT-Only**: Continual pre-training on food-science corpus
- **CPT→SFT**: Continual pre-training followed by supervised fine-tuning

In [ ]:
import json
import os
import glob as glob_module
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
from IPython.display import display, Markdown

# Plot style
matplotlib.rcParams.update({
    'figure.dpi': 150,
    'font.size': 10,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.facecolor': 'white',
})

CKPT_BASE = "/scratch/$USER/foodmole_workspace/checkpoints"
LOCAL_CACHE = "training_data.json"  # exported data for local use

# Experiment registry
experiments = {
    "Qwen3 SFT-Only":  {"path": f"{CKPT_BASE}/qwen3-8b-sft-lora-foodmole",       "color": "#2196F3", "stage": "SFT"},
    "Llama3 SFT-Only": {"path": f"{CKPT_BASE}/llama3-8b-sft-lora-foodmole",       "color": "#FF5722", "stage": "SFT"},
    "Qwen3 CPT":       {"path": f"{CKPT_BASE}/qwen3-8b-cpt",                      "color": "#4CAF50", "stage": "CPT"},
    "Llama3 CPT":      {"path": f"{CKPT_BASE}/llama3-8b-cpt",                      "color": "#FF9800", "stage": "CPT"},
    "Qwen3 CPT→SFT":  {"path": f"{CKPT_BASE}/qwen3-8b-cpt-sft-lora-foodmole",   "color": "#9C27B0", "stage": "CPT→SFT"},
    "Llama3 CPT→SFT": {"path": f"{CKPT_BASE}/llama3-8b-cpt-sft-lora-foodmole",   "color": "#E91E63", "stage": "CPT→SFT"},
}

def load_trainer_state(exp_path):
    """Load trainer_state.json, preferring top-level, fallback to latest checkpoint."""
    top_level = os.path.join(exp_path, "trainer_state.json")
    if os.path.exists(top_level):
        with open(top_level) as f:
            return json.load(f)
    ckpts = sorted(
        glob_module.glob(os.path.join(exp_path, "checkpoint-*/trainer_state.json")),
        key=lambda x: int(x.split("checkpoint-")[1].split("/")[0])
    )
    if ckpts:
        with open(ckpts[-1]) as f:
            return json.load(f)
    return None

def extract_loss(state):
    """Extract (steps, losses) from log_history."""
    if state is None:
        return [], []
    steps, losses = [], []
    for e in state.get("log_history", []):
        if "loss" in e and "step" in e:
            steps.append(e["step"])
            losses.append(e["loss"])
    return steps, losses

def extract_lr(state):
    """Extract (steps, learning_rate) from log_history."""
    if state is None:
        return [], []
    steps, lrs = [], []
    for e in state.get("log_history", []):
        if "learning_rate" in e and "step" in e:
            steps.append(e["step"])
            lrs.append(e["learning_rate"])
    return steps, lrs

# ============================================================
# Mode selection: LOCAL (from exported JSON) or HPC (from checkpoints)
# ============================================================
data = {}

if os.path.exists(LOCAL_CACHE):
    # --- LOCAL MODE: load from exported training_data.json ---
    print(f"Loading from local cache: {LOCAL_CACHE}")
    with open(LOCAL_CACHE) as f:
        data = json.load(f)
    for name in data:
        print(f"  {name}: {len(data[name]['steps'])} log entries, final_loss={data[name]['final_loss']:.4f}")
    print(f"\nLoaded {len(data)} experiments from local cache.")

else:
    # --- HPC MODE: load from checkpoint directories ---
    print(f"Loading from HPC checkpoints: {CKPT_BASE}")
    for name, cfg in experiments.items():
        state = load_trainer_state(cfg["path"])
        if state is not None:
            steps, losses = extract_loss(state)
            lr_steps, lrs = extract_lr(state)
            total_steps = state.get("max_steps", steps[-1] if steps else 0)
            data[name] = {
                "steps": steps, "losses": losses,
                "lr_steps": lr_steps, "lrs": lrs,
                "total_steps": total_steps,
                "final_loss": losses[-1] if losses else None,
                "best_loss": min(losses) if losses else None,
                "color": cfg["color"], "stage": cfg["stage"],
                "epoch": state.get("epoch", None),
                "train_runtime": state.get("train_runtime", None),
            }
            status = "✅ complete" if len(steps) >= total_steps * 0.98 else f"🔄 partial ({steps[-1]}/{total_steps})"
            print(f"  {name}: {len(steps)} log entries, final_loss={losses[-1]:.4f}, {status}")
        else:
            print(f"  {name}: ⏳ not started yet")
    print(f"\nLoaded {len(data)}/{len(experiments)} experiments from HPC.")

In [ ]:
# ============================================================
# Export data for local use (run this cell ONCE on HPC)
# Then download training_analysis.ipynb + training_data.json
# ============================================================
if not os.path.exists(LOCAL_CACHE) and data:
    with open(LOCAL_CACHE, "w") as f:
        json.dump(data, f)
    size_kb = os.path.getsize(LOCAL_CACHE) / 1024
    print(f"Exported {len(data)} experiments to {LOCAL_CACHE} ({size_kb:.0f} KB)")
    print("Download these 2 files to your local machine:")
    print(f"  1. training_analysis.ipynb")
    print(f"  2. {LOCAL_CACHE}")
else:
    if os.path.exists(LOCAL_CACHE):
        print(f"Local cache already exists: {LOCAL_CACHE}")
        print("Delete it and re-run to refresh from HPC checkpoints.")
    else:
        print("No data loaded — nothing to export.")

## 1. SFT Loss Curves

Supervised fine-tuning on `foodmole_sft_train.jsonl` (122,717 English food-science instruction pairs, 3 epochs).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Loss ---
for name in ["Qwen3 SFT-Only", "Llama3 SFT-Only"]:
    if name in data:
        d = data[name]
        ax1.plot(d["steps"], d["losses"], color=d["color"], linewidth=1.2,
                 label=f'{name} (final={d["final_loss"]:.3f})', alpha=0.85)

ax1.set_xlabel("Training Step")
ax1.set_ylabel("Loss")
ax1.set_title("SFT Training Loss")
ax1.legend(loc="upper right")

# --- Learning Rate ---
for name in ["Qwen3 SFT-Only", "Llama3 SFT-Only"]:
    if name in data:
        d = data[name]
        ax2.plot(d["lr_steps"], d["lrs"], color=d["color"], linewidth=1.2, label=name)

ax2.set_xlabel("Training Step")
ax2.set_ylabel("Learning Rate")
ax2.set_title("SFT Learning Rate Schedule")
ax2.legend(loc="upper right")

fig.suptitle("SFT-Only: Qwen3-8B vs Llama3.1-8B", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("sft_loss_curves.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: sft_loss_curves.png")

## 2. CPT Loss Curves

Continual pre-training on `cpt_corpus_merged.jsonl` (1.72M docs, ~3.12B tokens, 1 epoch).

Corpus composition by tokens: essay_fulltext 69.1% | fineweb_general 25.0% | essay_abstract 5.2% | wiki_food 0.7%

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for name in ["Qwen3 CPT", "Llama3 CPT"]:
    if name in data:
        d = data[name]
        # Subsample for readability if many points
        step = max(1, len(d["steps"]) // 2000)
        ax1.plot(d["steps"][::step], d["losses"][::step], color=d["color"], linewidth=0.8,
                 label=f'{name} (final={d["final_loss"]:.3f})', alpha=0.85)

ax1.set_xlabel("Training Step")
ax1.set_ylabel("Loss")
ax1.set_title("CPT Training Loss")
ax1.legend(loc="upper right")

for name in ["Qwen3 CPT", "Llama3 CPT"]:
    if name in data:
        d = data[name]
        step = max(1, len(d["lr_steps"]) // 2000)
        ax2.plot(d["lr_steps"][::step], d["lrs"][::step], color=d["color"], linewidth=0.8, label=name)

ax2.set_xlabel("Training Step")
ax2.set_ylabel("Learning Rate")
ax2.set_title("CPT Learning Rate Schedule")
ax2.legend(loc="upper right")

fig.suptitle("CPT: Qwen3-8B vs Llama3.1-8B", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("cpt_loss_curves.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: cpt_loss_curves.png")

## 3. CPT→SFT Loss Curves

Supervised fine-tuning initialized from CPT checkpoints (same SFT data & hyperparameters as SFT-Only).

In [ ]:
cpt_sft_names = [n for n in ["Qwen3 CPT→SFT", "Llama3 CPT→SFT"] if n in data]

if cpt_sft_names:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for name in cpt_sft_names:
        d = data[name]
        ax1.plot(d["steps"], d["losses"], color=d["color"], linewidth=1.2,
                 label=f'{name} (final={d["final_loss"]:.3f})', alpha=0.85)

    ax1.set_xlabel("Training Step")
    ax1.set_ylabel("Loss")
    ax1.set_title("CPT→SFT Training Loss")
    ax1.legend(loc="upper right")

    for name in cpt_sft_names:
        d = data[name]
        ax2.plot(d["lr_steps"], d["lrs"], color=d["color"], linewidth=1.2, label=name)

    ax2.set_xlabel("Training Step")
    ax2.set_ylabel("Learning Rate")
    ax2.set_title("CPT→SFT Learning Rate Schedule")
    ax2.legend(loc="upper right")

    fig.suptitle("CPT→SFT: Qwen3-8B vs Llama3.1-8B", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig("cpt_sft_loss_curves.png", dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved: cpt_sft_loss_curves.png")
else:
    print("⏳ No CPT→SFT experiments available yet.")

## 4. SFT Comparison: SFT-Only vs CPT→SFT

Side-by-side comparison of SFT loss curves with and without CPT pre-training, grouped by base model.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Qwen3 ---
for name, ls in [("Qwen3 SFT-Only", "-"), ("Qwen3 CPT→SFT", "--")]:
    if name in data:
        d = data[name]
        ax1.plot(d["steps"], d["losses"], color=d["color"], linewidth=1.2,
                 linestyle=ls, label=f'{name} (final={d["final_loss"]:.3f})', alpha=0.85)

ax1.set_xlabel("Training Step")
ax1.set_ylabel("Loss")
ax1.set_title("Qwen3-8B: SFT-Only vs CPT→SFT")
ax1.legend(loc="upper right")

# --- Llama3 ---
for name, ls in [("Llama3 SFT-Only", "-"), ("Llama3 CPT→SFT", "--")]:
    if name in data:
        d = data[name]
        ax2.plot(d["steps"], d["losses"], color=d["color"], linewidth=1.2,
                 linestyle=ls, label=f'{name} (final={d["final_loss"]:.3f})', alpha=0.85)

ax2.set_xlabel("Training Step")
ax2.set_ylabel("Loss")
ax2.set_title("Llama3.1-8B: SFT-Only vs CPT→SFT")
ax2.legend(loc="upper right")

fig.suptitle("Ablation: Effect of CPT Pre-training on SFT Convergence", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("sft_comparison.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: sft_comparison.png")

## 5. All Experiments Overview

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

stage_ls = {"SFT": "-", "CPT": "-.", "CPT→SFT": "--"}

for name, d in data.items():
    step = max(1, len(d["steps"]) // 2000)
    ls = stage_ls.get(d["stage"], "-")
    ax.plot(d["steps"][::step], d["losses"][::step], color=d["color"],
            linewidth=1.0, linestyle=ls, label=f'{name} (final={d["final_loss"]:.3f})', alpha=0.8)

ax.set_xlabel("Training Step")
ax.set_ylabel("Loss")
ax.set_title("All Training Runs — Loss Curves", fontsize=13, fontweight="bold")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.savefig("all_loss_curves.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: all_loss_curves.png")

## 6. Final Loss Summary

In [ ]:
# Bar chart of final losses
names = list(data.keys())
final_losses = [data[n]["final_loss"] for n in names]
colors = [data[n]["color"] for n in names]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(names, final_losses, color=colors, edgecolor="white", height=0.6)
ax.set_xlabel("Final Training Loss")
ax.set_title("Final Training Loss by Experiment", fontsize=13, fontweight="bold")
ax.invert_yaxis()

for bar, loss in zip(bars, final_losses):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{loss:.4f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("final_loss_summary.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: final_loss_summary.png")

---

## 7. Training Hyperparameters

All experiments use **LoRA** with the same adapter configuration.

In [ ]:
params_md = """
### LoRA Configuration (shared across all experiments)

| Parameter | Value |
|-----------|-------|
| LoRA Rank | 64 |
| LoRA Alpha | 128 |
| LoRA Target | all linear layers |
| LoRA Dropout | 0.05 |
| Precision | bf16 |
| Max Sequence Length | 2048 |
| LR Scheduler | cosine |
| Optimizer | adamw_torch |
| DDP Timeout | 180,000,000 ms |

### Per-Experiment Hyperparameters

| Experiment | Stage | Base Model | Batch/GPU | Grad Accum | Eff. Batch | GPUs | LR | Warmup | Epochs | Total Steps |
|------------|-------|------------|-----------|------------|------------|------|----|--------|--------|-------------|
| Qwen3 SFT-Only | SFT | Qwen3-8B-Base | 4 | 4 | 64 | 4×H200 | 2e-4 | 10% | 3 | 5,754 |
| Llama3 SFT-Only | SFT | Llama-3.1-8B | 2 | 8 | 64 | 4×H200 | 2e-4 | 10% | 3 | 5,754 |
| Qwen3 CPT | CPT | Qwen3-8B-Base | 2 | 8 | 64 | 4×H200 | 5e-5 | 1% | 1 | ~24,375 |
| Llama3 CPT | CPT | Llama-3.1-8B | 2 | 8 | 64 | 4×H200 | 5e-5 | 1% | 1 | ~24,375 |
| Qwen3 CPT→SFT | SFT | Qwen3-8B-Base + CPT adapter | 4 | 4 | 64 | 4×H200 | 2e-4 | 10% | 3 | 5,754 |
| Llama3 CPT→SFT | SFT | Llama-3.1-8B + CPT adapter | 2 | 8 | 64 | 4×H200 | 2e-4 | 10% | 3 | 5,754 |

**Note**: Llama3 uses batch_size=2 (vs Qwen3 batch_size=4) due to larger hidden_size (4096 vs 3840) causing OOM at batch=4.
Gradient accumulation is adjusted to maintain the same effective batch size of 64.
"""
display(Markdown(params_md))

## 8. Training Data

In [ ]:
data_md = """
### SFT Dataset: `foodmole_sft_train.jsonl`

| Property | Value |
|----------|-------|
| Format | Alpaca (`instruction`, `input`, `output`) |
| Language | English |
| Samples | 122,717 instruction-response pairs |
| Domain | Food science, nutrition, food safety, food processing |
| Max Length | cutoff_len = 2048 tokens |

### CPT Corpus: `cpt_corpus_merged.jsonl`

| Property | Value |
|----------|-------|
| Format | `{"text": "...", "source": "..."}` |
| Documents | 1,717,582 |
| Raw Size | 13.5 GB |
| Est. Tokens | ~3.12 billion |
| Max Length | cutoff_len = 2048 tokens |

**CPT corpus composition (by tokens)**:

| Source | Docs | % by Docs | % by Tokens | Description |
|--------|------|-----------|-------------|-------------|
| essay_fulltext | 253,756 | 14.8% | 69.1% | Full-text food science papers |
| fineweb_general | 1,006,276 | 58.6% | 25.0% | General domain web text (catastrophic forgetting prevention) |
| essay_abstract | 433,491 | 25.2% | 5.2% | Food science paper abstracts |
| wiki_food | 24,059 | 1.4% | 0.7% | Wikipedia food-related articles |

Effective ratio: **~75% food-science + ~25% general** (by tokens).
"""
display(Markdown(data_md))

## 9. Environment & Infrastructure

In [ ]:
env_md = """
### Hardware

| Component | Specification |
|-----------|---------------|
| Cluster | NUS Hopper HPC |
| GPU | NVIDIA H200 (80 GB HBM3) |
| GPUs per Job | 4 (single node) |
| Scheduler | PBS Pro |
| Interconnect | NVLink (intra-node) |

### Software Stack

| Component | Version |
|-----------|----------|
| Container | Singularity/Apptainer |
| Base Image | `pytorch_26.01-py3.sif` |
| Python | 3.12.3 |
| PyTorch | 2.10 |
| CUDA | 13.1 |
| LLaMA-Factory | 0.9.5.dev0 |
| Transformers | >=4.51.0 |
| PEFT | >=0.18.0 |
| Accelerate | >=1.3.0 |
| TRL | >=0.18.0 |
| Datasets | >=2.16.0 |

### Distributed Training

| Setting | Value |
|---------|-------|
| Strategy | DDP (DistributedDataParallel) via torchrun |
| Backend | NCCL |
| Launcher | LLaMA-Factory internal (`llamafactory-cli train`) |
| Nodes | 1 |
| Processes | 4 (1 per GPU) |
"""
display(Markdown(env_md))

## 10. Runtime Statistics

In [ ]:
# Runtime summary from trainer_state
rows = []
for name, d in data.items():
    runtime = d.get("train_runtime")
    if runtime:
        hours = runtime / 3600
        runtime_str = f"{hours:.1f}h"
    else:
        runtime_str = "(still running)"
    
    epoch = d.get("epoch")
    epoch_str = f"{epoch:.2f}" if epoch else "N/A"
    
    status = "✅" if d.get("train_runtime") else "🔄"
    
    rows.append(f"| {name} | {d['steps'][-1]:,} / {d['total_steps']:,} | {epoch_str} | {d['final_loss']:.4f} | {d['best_loss']:.4f} | {runtime_str} | {status} |")

table = """| Experiment | Steps | Epoch | Final Loss | Best Loss | Runtime | Status |
|------------|-------|-------|------------|-----------|---------|--------|
""" + "\n".join(rows)

display(Markdown(table))

---

*Generated by FoodmoleGPT Training Analysis Notebook | NUS Hopper HPC*